# Testing the Simplification of Regular Expressions

The notebook `Simplify.ipynb` provides the function `simplify`, which simplifies a regular expression, and the
function `to_str`, which turns a regular expression into a string that contains only the parentheses that are
needed.

In [ ]:
%run Simplify.ipynb

The notebook `RegExp-Parser.ipynb` provides the function `parse`, which turns a string into a regular
expression, and the notebook `09-Equivalence.ipynb` provides the function `regExpEquiv`, which checks whether
two regular expressions are equivalent.  We use it to check that `simplify` does not change the language of a
regular expression.

In [ ]:
%run RegExp-Parser.ipynb
%run 09-Equivalence.ipynb

**Note:** The type checker *basedpyright* analyzes every notebook on its own.  Since it cannot follow the
magic command `%run`, the calls of functions that are defined in the notebooks loaded above are marked with
`# type: ignore`.

In [ ]:
from typing import Literal
import random

In [ ]:
type BinaryOp = Literal['⋅', '+']
type UnaryOp  = Literal['*']
type RegExp   = int | str | tuple[RegExp, UnaryOp] | tuple[RegExp, BinaryOp, RegExp]

## Examples

Each of the following regular expressions is simplified by one of the laws discussed in the notebook
`Simplify.ipynb`.  The last example describes the complement of the language
$L\bigl((\texttt{a} + \texttt{b})^* \cdot \texttt{a} \cdot \texttt{b} \cdot (\texttt{a} + \texttt{b})^*\bigr)$.  It has been computed in the
notebook `12-Complement.ipynb` and simplified with the rules from the notebook `Rewrite.ipynb`.

In [ ]:
Σ = { 'a', 'b' }
for s in [ 'b + a + b*', '(𝜀 + a + b)*', '(a*b*)*', 'ab + aa', 'a + ab*', 'a*(a + 𝜀)', 'b*aa* + 𝜀 + b + b*' ]:
    r = parse(s)     # type: ignore
    t = simplify(r)  # type: ignore
    print(f'{to_str(r):24} ↦ {to_str(t):12} {regExpEquiv(r, t, Σ)}')  # type: ignore

## Random Tests

In order to test the function `simplify` more thoroughly, we generate random regular expressions and check that
`simplify` does not change their language.

**Function `random_regexp(n, rng)`**
- *Input:* `n` is a natural number and `rng` is a random number generator.
- *Output:* A random regular expression over the alphabet $\{\texttt{a}, \texttt{b}\}$ that contains exactly `n`
  operators.  Its atoms are chosen from $\texttt{a}$, $\texttt{b}$, $\varepsilon$, and $\emptyset$.

In [ ]:
def random_regexp(n: int, rng: random.Random) -> RegExp:
    if n == 0:
        return rng.choice(['a', 'b', '𝜀', 0])
    match rng.choice(['+', '⋅', '*']):
        case '*':
            return (random_regexp(n - 1, rng), '*')
        case '+':
            k = rng.randint(0, n - 1)
            return (random_regexp(k, rng), '+', random_regexp(n - 1 - k, rng))
        case _:
            k = rng.randint(0, n - 1)
            return (random_regexp(k, rng), '⋅', random_regexp(n - 1 - k, rng))

We test `simplify` with 3000 random regular expressions that have up to 14 operators.  The random number
generator is initialized with a fixed seed, so that the test is reproducible.  The test prints every regular
expression whose simplification is not equivalent to it, the number of these failures, and the total sizes of
the regular expressions before and after simplification.

In [ ]:
rng      = random.Random(42)
failures = 0
before   = 0
after    = 0
for _ in range(3000):
    r = random_regexp(rng.randint(1, 14), rng)
    t = simplify(r)  # type: ignore
    before += size(r)  # type: ignore
    after  += size(t)  # type: ignore
    if not regExpEquiv(r, t, Σ):  # type: ignore
        failures += 1
        print(f'{to_str(r)} ↦ {to_str(t)}')  # type: ignore
print(f'failures: {failures}, total size before: {before}, after: {after}')